# Modelo de hipótesis.

### Hipótesis.

Creemos que un modelo que implemente:
- Una red CNN-1D para identificar las transacciones cortas.
- Una red LSTM para identificar aquellas anomalías en el orden de las transacciones.
- Random Forest que permite identificar las anomalías por tipo de negocio.

mejorará la predicción de las anomalías, debido a que cada sub-modelo estará especializado en un tipo de anomalía porque es más específico y aprende a reconocer los patrones que surgen de las anomalías consideradas.

Lo consideraremos útil si cada sub-modelo logra obtener un alto pr-auc para las anomalías con las que fue entrenado, por esta razón, si por lo menos un sub-modelo reconoce una anomalía, el modelo general lo tomará como anomalía.

### Importación de librerías.

In [1]:
import numpy as np
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import average_precision_score, precision_recall_fscore_support, precision_recall_curve, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin

### Importación de datos.

In [2]:
df = pd.read_csv('datos_sinteticos.csv', sep=";")
df.head(2)

,cliente_id,fechas_hora,montos,tipos_establecimiento,historico_establecimiento,es_anomalia,tipo_anomalia,promedio,desviacion,maximo,minimo,monto_hora,moda_establecimiento,cantidad_establecimiento
0,1,"['2026-06-08 00:16:00', '2026-06-08 06:14:00',...","['705.29', '628.12', '272.24', '382.26', '1109...","['Farmacia', 'Farmacia', 'Transporte', 'Farmac...","['0.16643057183060853', '0.017594963536259', '...",0,Normal,879.374333,602.582962,2380.04,240.25,34.208766,Farmacia,7
1,2,"['2026-06-14 07:11:00', '2026-06-15 07:48:00',...","['266.04', '518.87', '266.71', '193.6', '264.7...","['Otros', 'Otros', 'Transporte', 'Transporte',...","['0.001894439046728334', '0.01438288653067728'...",0,Normal,346.078000,165.763599,762.78,63.60,16.192467,Otros,5


### Eliminando variables no útiles.

In [3]:
df = df.drop(columns=['cliente_id', 'promedio', 'desviacion', 'maximo', 'minimo', 'monto_hora', 'moda_establecimiento', 'cantidad_establecimiento'])
df.head(2)

,fechas_hora,montos,tipos_establecimiento,historico_establecimiento,es_anomalia,tipo_anomalia
0,"['2026-06-08 00:16:00', '2026-06-08 06:14:00',...","['705.29', '628.12', '272.24', '382.26', '1109...","['Farmacia', 'Farmacia', 'Transporte', 'Farmac...","['0.16643057183060853', '0.017594963536259', '...",0,Normal
1,"['2026-06-14 07:11:00', '2026-06-15 07:48:00',...","['266.04', '518.87', '266.71', '193.6', '264.7...","['Otros', 'Otros', 'Transporte', 'Transporte',...","['0.001894439046728334', '0.01438288653067728'...",0,Normal


### Separando las variables predictoras de la objetivo.

In [4]:
X = df.drop(columns = ['es_anomalia'])
y = df['es_anomalia']

### Separación del conjunto de prueba y entrenamiento.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0, stratify=X['tipo_anomalia'])

### Separación de los datos para diferentes modelos.

#### Datos CNN-1D.

In [6]:
X_cnn_train = X_train[((X_train['tipo_anomalia'] == 'transaccion_corta') | (X_train['tipo_anomalia'] == 'Normal'))][['fechas_hora', 'tipo_anomalia']].reset_index()
X_cnn_test = X_test[((X_test['tipo_anomalia'] == 'transaccion_corta') | (X_test['tipo_anomalia'] == 'Normal'))][['fechas_hora', 'tipo_anomalia']].reset_index()

y_cnn_train = y_train[((X_train['tipo_anomalia'] == 'transaccion_corta') | (X_train['tipo_anomalia'] == 'Normal'))]
y_cnn_test = y_test[((X_test['tipo_anomalia'] == 'transaccion_corta') | (X_test['tipo_anomalia'] == 'Normal'))]

X_cnn_train = X_cnn_train.drop(columns = ['tipo_anomalia'])
X_cnn_test = X_cnn_test.drop(columns = ['tipo_anomalia'])

In [7]:
print(X_cnn_train.shape, y_cnn_train.shape)
print(X_cnn_test.shape, y_cnn_test.shape)

(74838, 2) (74838,)
(18709, 2) (18709,)


#### Datos LSTM.

In [8]:
X_lstm_train = X_train[((X_train['tipo_anomalia'] == 'orden_no_congruente') | (X_train['tipo_anomalia'] == 'Normal'))][['montos', 'tipo_anomalia']].reset_index()
X_lstm_test = X_test[((X_test['tipo_anomalia'] == 'orden_no_congruente') | (X_test['tipo_anomalia'] == 'Normal'))][['montos', 'tipo_anomalia']].reset_index()

y_lstm_train = y_train[((X_train['tipo_anomalia'] == 'orden_no_congruente') | (X_train['tipo_anomalia'] == 'Normal'))]
y_lstm_test = y_test[((X_test['tipo_anomalia'] == 'orden_no_congruente') | (X_test['tipo_anomalia'] == 'Normal'))]

X_lstm_train = X_lstm_train.drop(columns = ['tipo_anomalia'])
X_lstm_test = X_lstm_test.drop(columns = ['tipo_anomalia'])

In [9]:
print(X_lstm_train.shape, y_lstm_train.shape)
print(X_lstm_test.shape, y_lstm_test.shape)

(75144, 2) (75144,)
(18786, 2) (18786,)


#### Datos Random Forest.

In [10]:
X_rf_train = X_train[((X_train['tipo_anomalia'] == 'establecimiento_raro') | (X_train['tipo_anomalia'] == 'Normal'))][['tipos_establecimiento', 'historico_establecimiento', 'tipo_anomalia']].reset_index()
X_rf_test = X_test[((X_test['tipo_anomalia'] == 'establecimiento_raro') | (X_test['tipo_anomalia'] == 'Normal'))][['tipos_establecimiento', 'historico_establecimiento', 'tipo_anomalia']].reset_index()

y_rf_train = y_train[((X_train['tipo_anomalia'] == 'establecimiento_raro') | (X_train['tipo_anomalia'] == 'Normal'))]
y_rf_test = y_test[((X_test['tipo_anomalia'] == 'establecimiento_raro') | (X_test['tipo_anomalia'] == 'Normal'))]

X_rf_train = X_rf_train.drop(columns = ['tipo_anomalia'])
X_rf_test = X_rf_test.drop(columns = ['tipo_anomalia'])

### Preprocesamiento preliminar de datos (no del modelo).

#### Datos CNN-1D.

In [11]:
def convertir_fechas_a_minutos(columna_fechas):
    # Separa las 30 fechas en columnas.
    fechas_texto = (columna_fechas.astype("string").str.strip().str.slice(1, -1).str.replace("'", "", regex=False).str.split(r",\s*", expand=True, regex=True))

    # El formato explícito evita que Pandas intente inferirlo.
    fechas_datetime = fechas_texto.apply(pd.to_datetime, format="%Y-%m-%d %H:%M:%S", errors="raise")

    # Matriz de numpy.
    fechas_numpy = fechas_datetime.to_numpy(dtype="datetime64[ns]")

    # Diferencia en minutos entre transacciones consecutivas.
    diferencias_minutos = (np.diff(fechas_numpy, axis=1) / np.timedelta64(1, "m"))

    return diferencias_minutos.astype(np.float32)

In [12]:
pipeline_fechas = Pipeline(steps=[
    ("diferencias_minutos", FunctionTransformer(convertir_fechas_a_minutos, validate=False)),
    ("transformacion_logaritmica", FunctionTransformer(np.log1p, validate=False)),
    ("normalizacion", StandardScaler(copy=True, with_mean=True, with_std=True))
    ], verbose=False)

#### Datos LSTM.

In [13]:
def convertir_montos(datos):

    # Se genera una serie.
    if isinstance(datos, pd.DataFrame):
        columna = datos.iloc[:, 0]
    else:
        columna = datos

    # Se separan los datos para convertirlos en listas de números.
    montos_separados = (columna.astype("string").str.strip().str.slice(1, -1).str.replace("'", "", regex=False).str.split(r",\s*", expand=True, regex=True))

    return montos_separados.to_numpy(dtype=np.float32)

In [14]:
class EscaladorGlobalSecuencias(BaseEstimator, TransformerMixin):

    # Entrenador del modelo.
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=np.float32)

        self.media_ = X.mean()
        self.desviacion_ = X.std(ddof=0)

        if self.desviacion_ == 0:
            self.desviacion_ = 1.0

        return self

    # Transformador del modelo.
    def transform(self, X):
        X = np.asarray(X, dtype=np.float32)
        return ((X - self.media_) / self.desviacion_).astype(np.float32)

In [15]:
pipeline_montos = Pipeline(steps=[
    ("convertir_a_numerico", FunctionTransformer(convertir_montos, validate=False)),
    ("transformacion_logaritmica", FunctionTransformer(np.log1p, validate=False)),
    ("normalizacion_global", EscaladorGlobalSecuencias())
    ], verbose=False)

#### Datos Random Forest.

In [16]:
ESTABLECIMIENTOS = ["Supermercado", "Restaurante", "Gasolinera", "Farmacia", "Ropa",
                    "Tecnología", "Entretenimiento", "Transporte", "Otros"]

In [37]:
def calcular_diferencias_establecimientos(datos):

    datos = datos.reset_index(drop=True).copy()

    # Convierte el string en una lista de establecimientos.
    tipos = (datos["tipos_establecimiento"].astype("string").str.strip().str.slice(1, -1).str.replace("'", "", regex=False).str.split(r",\s*", regex=True))

    # Lleva las listas a formato largo.
    tipos_largos = tipos.explode().str.strip()

    # Cuenta cuántas veces aparece cada establecimiento por fila.
    frecuencias_actuales = pd.crosstab(index=tipos_largos.index, columns=tipos_largos)

    # Garantiza el orden exacto de las nueve categorías.
    frecuencias_actuales = frecuencias_actuales.reindex(index=datos.index, columns=ESTABLECIMIENTOS, fill_value=0)

    # Convierte los conteos en frecuencias relativas.
    cantidades_transacciones = tipos.str.len()
    frecuencias_actuales = frecuencias_actuales.div(cantidades_transacciones, axis=0)

    # Convierte el histórico de string a una matriz numérica.
    frecuencias_historicas = (datos["historico_establecimiento"].astype("string").str.strip().str.slice(1, -1).str.replace("'", "", regex=False).str.split(r",\s*", expand=True, regex=True))
    frecuencias_historicas = frecuencias_historicas.astype(np.float32)
    frecuencias_historicas.columns = ESTABLECIMIENTOS

    # Diferencias absolutas entre frecuencia actual e histórica.
    diferencias = np.abs(frecuencias_actuales.to_numpy(dtype=np.float32) - frecuencias_historicas.to_numpy(dtype=np.float32))
    nombres_columnas = [f"diferencia_{establecimiento.lower()}" for establecimiento in ESTABLECIMIENTOS]

    # DataFrame con la diferencia de frecuencias.
    df_nuevo = pd.DataFrame(diferencias, columns=nombres_columnas, index=datos.index, dtype=np.float32)

    # Se guardan los históricos.
    for x in range(len(ESTABLECIMIENTOS)):
        df_nuevo['historico_' + ESTABLECIMIENTOS[x]] = frecuencias_historicas[ESTABLECIMIENTOS[x]]
    
    return df_nuevo

In [39]:
pipeline_establecimientos = Pipeline(steps=[
    ("diferencias_establecimientos", FunctionTransformer( calcular_diferencias_establecimientos, validate=False))], verbose=False)

## CNN-1D

### Separación del conjunto de entrenamiento y de validación.

In [19]:
cnn_train_x, cnn_val_x, cnn_train_y, cnn_val_y = train_test_split(X_cnn_train, y_cnn_train, test_size=0.20, random_state=0, stratify=y_cnn_train)

### Transformación de datos.

In [20]:
# Preprocesando los datos.
cnn_train_x = pipeline_fechas.fit_transform(cnn_train_x['fechas_hora'])
cnn_val_x = pipeline_fechas.transform(cnn_val_x['fechas_hora'])
cnn_test_x = pipeline_fechas.transform(X_cnn_test['fechas_hora'])

# Conv1D espera (n, 29, 1).
cnn_train_x = cnn_train_x.astype(np.float32)[..., np.newaxis]
cnn_val_x = cnn_val_x.astype(np.float32)[..., np.newaxis]
cnn_test_x = cnn_test_x.astype(np.float32)[..., np.newaxis]

# Cambiando la estructura de las etiquetas
cnn_train_y = np.asarray(cnn_train_y, dtype=np.float32).reshape(-1)
cnn_val_y = np.asarray(cnn_val_y, dtype=np.float32).reshape(-1)
cnn_test_y = np.asarray(y_cnn_test, dtype=np.float32).reshape(-1)

### Arquitectura de datos.

In [21]:
tf.keras.utils.set_random_seed(0)

modelo_cnn = tf.keras.Sequential([

        # Capa de entrada.
        tf.keras.layers.Input(shape=(29, 1)),

        # Primer bloque.
        tf.keras.layers.Conv1D(filters=32, kernel_size=3, strides=1, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2, strides=2),

        # Segundo bloque.
        tf.keras.layers.Conv1D(filters=64, kernel_size=3, strides=1, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalMaxPooling1D(),

        # Capa densa.
        tf.keras.layers.Dense(units=32, activation="relu"),
        tf.keras.layers.Dropout(rate=0.10),

        # Clasificación binaria.
        tf.keras.layers.Dense(units=1, activation="sigmoid")
    ], name="cnn_fechas")

### Compilacón del modelo.

In [22]:
modelo_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5),
             tf.keras.metrics.Precision(name="precision", thresholds=0.5),
             tf.keras.metrics.Recall(name="recall", thresholds=0.5),
             tf.keras.metrics.AUC(name="auc", curve="ROC"),
             tf.keras.metrics.AUC(name="pr_auc", curve="PR")])

modelo_cnn.summary()

Model: "cnn_fechas"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 29, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 29, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 14, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 14, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,833 (34.50 KB)

 Trainable params: 8,641 (33.75 KB)

 Non-trainable params: 192 (768.00 B)

### Balanceo de clases.

In [23]:
clases = np.unique(y_train)
pesos = compute_class_weight(class_weight="balanced", classes=clases, y=cnn_train_y)
class_weight = {int(clase): float(peso) for clase, peso in zip(clases, pesos)}

print("Pesos de las clases:", class_weight)

Pesos de las clases: {0: 0.5200750534234437, 1: 12.953266983989614}


### Entrenamiento del modelo.

In [24]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=8, min_delta=1e-4, restore_best_weights=True, verbose=1),
             tf.keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc",mode="max", factor=0.5, patience=3, min_lr=1e-6, verbose=1)]

historial = modelo_cnn.fit(
    cnn_train_x,
    cnn_train_y,
    validation_data=(cnn_val_x, cnn_val_y),
    epochs=30,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

Epoch 1/30
936/936 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9719 - auc: 0.9969 - loss: 0.0643 - pr_auc: 0.9158 - precision: 0.5802 - recall: 0.9849 - val_accuracy: 0.9916 - val_auc: 0.9992 - val_loss: 0.0277 - val_pr_auc: 0.9672 - val_precision: 0.8219 - val_recall: 0.9983 - learning_rate: 0.0010
Epoch 2/30
936/936 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9880 - auc: 0.9986 - loss: 0.0288 - pr_auc: 0.9436 - precision: 0.7651 - recall: 0.9948 - val_accuracy: 0.9933 - val_auc: 0.9995 - val_loss: 0.0217 - val_pr_auc: 0.9812 - val_precision: 0.8536 - val_recall: 0.9983 - learning_rate: 0.0010
Epoch 3/30
936/936 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9914 - auc: 0.9990 - loss: 0.0213 - pr_auc: 0.9597 - precision: 0.8195 - recall: 0.9961 - val_accuracy: 0.9950 - val_auc: 0.9994 - val_loss: 0.0182 - val_pr_auc: 0.9741 - val_precision: 0.8863 - val_recall: 0.9983 - learning_rate: 0.0010
Epoch 4/30
936/936 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9915 - auc: 0.9990 - l

### Medición del desempeño en conjunto de prueba.

In [25]:
resultados = modelo_cnn.evaluate(cnn_test_x, cnn_test_y, batch_size=256, verbose=0, return_dict=True)

print("Resultados de validación:")
for metrica, valor in resultados.items():
    print(f"{metrica}: {valor:.4f}")

Resultados de validación:
accuracy: 0.9950
auc: 0.9997
loss: 0.0144
pr_auc: 0.9906
precision: 0.8857
recall: 0.9986


### Guardando el modelo y el pipeline.

In [26]:
# Modelo
modelo_cnn.save("modelo_c_cnn.keras")

# Pipeline
joblib.dump(pipeline_fechas, "pipeline_c_cnn.joblib")

['pipeline_c_cnn.joblib']

## LSTM.

### Separación del conjunto de entrenamiento y de validación.

In [27]:
lstm_train_x, lstm_val_x, lstm_train_y, lstm_val_y = train_test_split(X_lstm_train, y_lstm_train, test_size=0.20, random_state=0, stratify=y_lstm_train)

### Transformación de datos.

In [ ]:
# Preprocesando los datos.
lstm_train_x = pipeline_montos.fit_transform(lstm_train_x['montos'])
lstm_val_x = pipeline_montos.transform(lstm_val_x['montos'])
lstm_test_x = pipeline_montos.transform(X_lstm_test['montos'])

# Conv1D espera (n, 30, 1).
lstm_train_x = lstm_train_x.astype(np.float32)[..., np.newaxis]
lstm_val_x = lstm_val_x.astype(np.float32)[..., np.newaxis]
lstm_test_x = lstm_test_x.astype(np.float32)[..., np.newaxis]

# Cambiando la estructura de las etiquetas
lstm_train_y = np.asarray(lstm_train_y, dtype=np.float32).reshape(-1)
lstm_val_y = np.asarray(lstm_val_y, dtype=np.float32).reshape(-1)
lstm_test_y = np.asarray(y_lstm_test, dtype=np.float32).reshape(-1)

### Arquitectura de datos.

In [29]:
tf.keras.utils.set_random_seed(0)

modelo_lstm = tf.keras.Sequential([

    # Capa de entrada.
    tf.keras.layers.Input(shape=(30, 1), name="secuencia_montos"),

    # Primer bloque.
    tf.keras.layers.LSTM(units=64, activation="tanh", recurrent_activation="sigmoid", return_sequences=True,
                         dropout=0.0, recurrent_dropout=0.0, name="lstm_1"),
    tf.keras.layers.LayerNormalization(name="normalizacion_lstm_1"),
    tf.keras.layers.Dropout(rate=0.1, name="dropout_1"),

    # Segundo bloque.
    tf.keras.layers.LSTM(units=32, activation="tanh", recurrent_activation="sigmoid", return_sequences=False,
                         dropout=0.0, recurrent_dropout=0.0, name="lstm_2"),
    tf.keras.layers.Dense(units=32, activation="relu", name="capa_densa"),
    #tf.keras.layers.Dropout(rate=0.1, name="dropout_2"),

    # Capa densa.
    tf.keras.layers.Dense(units=1, activation="sigmoid", name="score_fraude")

    ], name="lstm_montos"
)

### Compilacón del modelo.

In [30]:
modelo_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5),
             tf.keras.metrics.Precision(name="precision", thresholds=0.5),
             tf.keras.metrics.Recall(name="recall", thresholds=0.5),
             tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
             tf.keras.metrics.AUC(name="pr_auc", curve="PR")])

modelo_lstm.summary()

Model: "lstm_montos"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 30, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalizacion_lstm_1            │ (None, 30, 64)         │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_densa (Dense)              │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ score_fraude (Dense)            │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,529 (119.25 KB)

 Trainable params: 30,529 (119.25 KB)

 Non-trainable params: 0 (0.00 B)

### Balanceo de clases.

In [31]:
clases = np.unique(lstm_train_y)
pesos = compute_class_weight(class_weight="balanced", classes=clases, y=lstm_train_y)
class_weight = {int(clase): float(peso) for clase, peso in zip(clases, pesos)}

print("Pesos de las clases:", class_weight)

Pesos de las clases: {0: 0.5222033044354488, 1: 11.759585289514867}


### Entrenamiento del modelo.

In [32]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=8, min_delta=1e-4, restore_best_weights=True, verbose=1),
             tf.keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc", mode="max", factor=0.5, patience=3, min_delta=1e-4, min_lr=1e-6, verbose=1)]

historial_lstm = modelo_lstm.fit(
    lstm_train_x,
    lstm_train_y,
    validation_data=(lstm_val_x, lstm_val_y),
    epochs=30,
    batch_size=256,
    class_weight=class_weight,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

Epoch 1/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 35s 139ms/step - accuracy: 0.6499 - loss: 0.5972 - pr_auc: 0.1332 - precision: 0.0779 - recall: 0.6674 - roc_auc: 0.7351 - val_accuracy: 0.8337 - val_loss: 0.4078 - val_pr_auc: 0.3757 - val_precision: 0.1856 - val_recall: 0.8592 - val_roc_auc: 0.9168 - learning_rate: 0.0010
Epoch 2/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.8814 - loss: 0.2891 - pr_auc: 0.5124 - precision: 0.2491 - recall: 0.8877 - roc_auc: 0.9485 - val_accuracy: 0.9480 - val_loss: 0.1516 - val_pr_auc: 0.6959 - val_precision: 0.4426 - val_recall: 0.8560 - val_roc_auc: 0.9724 - learning_rate: 0.0010
Epoch 3/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.9077 - loss: 0.2145 - pr_auc: 0.6322 - precision: 0.3055 - recall: 0.9194 - roc_auc: 0.9707 - val_accuracy: 0.9342 - val_loss: 0.1547 - val_pr_auc: 0.7502 - val_precision: 0.3852 - val_recall: 0.9186 - val_roc_auc: 0.9814 - learning_rate: 0.0010
Epoch 4/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accu

### Medición del desempeño en conjunto de prueba.

In [33]:
resultados = modelo_lstm.evaluate(lstm_test_x, lstm_test_y, batch_size=256, verbose=0, return_dict=True)

for metrica, valor in resultados.items():
    print(f"{metrica}: {valor:.4f}")

accuracy: 0.9775
loss: 0.0610
pr_auc: 0.9234
precision: 0.6593
recall: 0.9762
roc_auc: 0.9960


### Guardando el modelo y el pipeline.

In [ ]:
# Modelo
modelo_lstm.save("modelo_c_lstm.keras")

# Pipeline
joblib.dump(pipeline_montos, "pipeline_c_lstm.joblib")

['pipeline_lstm_montos.joblib']

## Random Forest.

### Pipeline.

In [40]:
modelo_rf = Pipeline(
    steps=[("preprocesamiento", pipeline_establecimientos),
           ("random_forest", RandomForestClassifier(criterion="gini", bootstrap=True, random_state=0, n_jobs=1))
           ], verbose=False)

### Rejilla de hiperparámetros.

In [41]:
param_grid = {
    "random_forest__n_estimators": [25, 50, 100],
    "random_forest__max_depth": [3, 5, 7],
    "random_forest__min_samples_leaf": [1, 3, 6],
    "random_forest__max_features": ["sqrt", 0.70],
    "random_forest__class_weight": ["balanced_subsample"]
}

### GridSearch.

In [42]:
cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

grid_search_rf = GridSearchCV(
    estimator=modelo_rf,
    param_grid=param_grid,
    scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc"},
    refit="pr_auc",
    cv=cv_estratificado,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    error_score="raise"
)

### Entrenamiento del modelo.

In [43]:
grid_search_rf.fit(X_rf_train, y_rf_train)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


,estimator,Pipeline(step...om_state=0))])
,param_grid,"{'random_forest__class_weight': ['balanced_subsample'], 'random_forest__max_depth': [3, 5, ...], 'random_forest__max_features': ['sqrt', 0.7], 'random_forest__min_samples_leaf': [1, 3, ...], ...}"
,scoring,"{'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'}"
,n_jobs,-1
,refit,'pr_auc'
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,'raise'
,return_train_score,True
,steps,"[('diferencias_establecimientos', ...)]"


### Mejor modelo.

In [44]:
print(f"Mejor PR-AUC promedio en CV: " f"{grid_search_rf.best_score_:.4f}")
print("\nMejores hiperparámetros:")

for parametro, valor in grid_search_rf.best_params_.items():
    print(f"{parametro}: {valor}")

mejor_modelo_rf = grid_search_rf.best_estimator_

Mejor PR-AUC promedio en CV: 0.6287

Mejores hiperparámetros:
random_forest__class_weight: balanced_subsample
random_forest__max_depth: 7
random_forest__max_features: 0.7
random_forest__min_samples_leaf: 6
random_forest__n_estimators: 100


### Medición del desempeño en conjunto de prueba.

In [45]:
probabilidades_val_rf = mejor_modelo_rf.predict_proba(X_rf_test)[:, 1]

pr_auc_val = average_precision_score(y_rf_test, probabilidades_val_rf)
roc_auc_val = roc_auc_score(y_rf_test, probabilidades_val_rf)

print(f"PR-AUC prueba:  {pr_auc_val:.4f}")
print(f"ROC-AUC prueba: {roc_auc_val:.4f}")

PR-AUC prueba:  0.6397
ROC-AUC prueba: 0.9744


In [46]:
predicciones_val_rf = (probabilidades_val_rf >= 0.5).astype(np.int32)
print(classification_report(y_rf_test, predicciones_val_rf, digits=4, zero_division=0))

              precision    recall  f1-score   support

           0     0.9940    0.9717    0.9827     17987
           1     0.4313    0.7846    0.5566       492

    accuracy                         0.9667     18479
   macro avg     0.7126    0.8781    0.7697     18479
weighted avg     0.9790    0.9667    0.9714     18479



### Guardando el modelo y el pipeline.

In [47]:
joblib.dump(mejor_modelo_rf, "modelo_c_rf.joblib", compress=3)

['modelo_c_rf.joblib']